In [ ]:
import plotly.express as px
import torch
from darts.utils.statistics import plot_ccf, remove_seasonality
from darts.utils.utils import SeasonalityMode

from aare.constants import TIME
from aare.params import read_params
from aare.preparation import resample, interpolate
from aare.remote_existenz_store import RemoteExistenzStore
from aare.utils import to_ts

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
# logging.basicConfig(level="DEBUG")

In [ ]:
params = read_params()
store = RemoteExistenzStore()

In [ ]:
ANYTIME = "0"  # to be used as period start when querying influx. starting at 0 just returns all the data.

# Variables to look at

<https://api-datasette.konzept.space/existenz-api/hydro_parameters> \
<https://api-datasette.konzept.space/existenz-api/smn_parameters>

- Flow (hydro/flow):
  - Does the flow or change in flow have an influence on the temperature or change in temperature?
- Precipitation (smn/rr):
  - Does the precipitation have an influence on the temperature or change in temperature?
  - Does the precipitation have an influence on the flow? Probably yes, but at which lag is the correlation greatest?
  - To be useful as feature, pool together a total over some hours and make sure the model has access to all relevant lags.
- Sunshine duration (smn/ss):
  - Does the sunshine duration have an influence on the temperature or change in temperature?
  - How much lag is there in the heat transfer from the sunshine to the water temperature? could influence water directly without air temperature
- Air temperature (smn/tt):
  - How high is the correlation between air temperature and water temperature?
  - How much lag is there in the heat transfer from the air temperature to the water temperature?
  - Does the daily, weekly or monthly mean show correlation to the change in water temperature? might be nice feature
- Turbidity (hydro/turbidity):
  - Might influence the transfer rate from sunshine to water temperature
- Global Exposure (smn/rad):
  - Don't quite know what this is and does.
  - Probably just affects air temperature and won't make a good feature but idk.
- Relative Humidity (smn/rh):
  - Does humidity have any direct influence on the water temperature? If there is correlation, I suspect it's just because air temperature and precipitation affect RH.
  - Probably not a good feature

 Unsure of confounders:
 - Precipitation affects flow
 - Sunshine affects air temperature
 - Turbidity is affected by flow and precipitation

### Note on lags

If these features are to be used for forecasting, you must make sure that either

1. their influence delay is larger than our forecast horizon (if it's delayed > 4 days, can use data from now to predict 4 days into the future)
1. there is a forecast available for the data (we can use existing/official forecasts for air temperature, precipitation and flow to help forecasts further out)
1. we are also forecasting that variable to be able to use it autoregressively for future forecasts

If none of these are true, we cannot use it to forecast for the desired horizon and need to shorten, drop the feature or forecast it ourselves (bullet 3).

### Note on frequency

For some features, like the precipitation, we probably don't want the average of 10 min totals over an hour (agg mean) but rather the total over an hour (agg sum).
Same goes for sunshine duration for example.

### Note on availability

Not all of those features might be available as far back as the water temperature.
If we find that some feature would be really nice for training, but it's only available for the last few years,^
we might need to consider using less data for training and validation. If performance is not good enough, could think about pre-training.

In [ ]:
df = store.query(
    "-1y",
    [
        "hydro/temperature:mean_1h@bern",
        "hydro/temperature:mean_1h@thun",
        "hydro/flow:mean_1h@bern",
        "hydro/flow:mean_1h@thun",
        "smn/rr:sum_1h@bern",
        "smn/rr:sum_1h@thun",
        "smn/ss:sum_1h@bern",
        "smn/rad:sum_1h@bern",
        # "hydro/turbidity:mean_1h@bern" <- not returned apparently? must investigate
        "smn/tt:mean_1h@bern",
    ],
)
df

In [ ]:
df = resample(df)
df

In [ ]:
df.isna().sum()

In [ ]:
df = interpolate(df, drop_filled=True, columns=None)
df

In [ ]:
df.isna().sum()

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "temperature_thun"])

In [ ]:
temp_bern = to_ts(df, col="temperature_bern")
temp_thun = to_ts(df, col="temperature_thun")

In [ ]:
plot_ccf(temp_bern, temp_thun, max_lag=4 * 24)

In [ ]:
plot_ccf(remove_seasonality(temp_bern, freq=24), remove_seasonality(temp_thun, freq=24), max_lag=4 * 24)

In [ ]:
plot_ccf(
    remove_seasonality(temp_bern, freq=24, method="STL", model=SeasonalityMode.ADDITIVE),
    remove_seasonality(temp_thun, freq=24, method="STL", model=SeasonalityMode.ADDITIVE),
    max_lag=4 * 24,
)